In [1]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [2]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [4]:
import json

def make_call(call):
    args = json.loads(call.arguments)
    print(args)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [18]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]


In [5]:
from openai import OpenAI
openai_client = OpenAI()

In [6]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

In [15]:
messages.extend(response.output)
has_function_calls = False

for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

function_call: search {"query":"join course enrollment discovered the course can I join"}
{'query': 'join course enrollment discovered the course can I join'}
function_call: search {"query":"course registration late enrollment can I join after start"}
{'query': 'course registration late enrollment can I join after start'}
function_call: search {"query":"enroll in course after discovering course access signup"}
{'query': 'enroll in course after discovering course access signup'}


In [16]:
messages

[{'role': 'developer',
  'content': "You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore."},
 {'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"join course enrollment discovered the course can I join"}', call_id='call_fyYAQgPDEzd6hbVgMY5prFdV', name='search', type='function_call', id='fc_0b6712a1ef7a2880006a381d30f3b081a2938b53dbc2727229', namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"course registration late enrollment can I join after start"}', call_id='call_Ncwsci7ChMf

In [19]:
it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    it = it + 1
    if has_function_calls == False:
        break

iteration #1...
function_call: search {"query":"join the course discovered course can I join enrollment registration late join FAQ"}
{'query': 'join the course discovered course can I join enrollment registration late join FAQ'}
function_call: search {"query":"course enrollment can I join after course started discovered the course FAQ"}
{'query': 'course enrollment can I join after course started discovered the course FAQ'}
iteration #2...
ASSISTANT:
Yes, you can still join the course.

If you want a certificate, make sure you submit your project while submissions are still open. You can also start learning right away from the course materials.

If you’d like, I can also help with how to get started or explain the certificate requirements.


In [20]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [21]:
agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
function_call: search {"query":"Ollama run locally install local model macOS Windows Linux how do I run Ollama locally"}
{'query': 'Ollama run locally install local model macOS Windows Linux how do I run Ollama locally'}
iteration #2...
function_call: search {"query":"ollama serve localhost 11434 run llama3 local server Python client course FAQ"}
{'query': 'ollama serve localhost 11434 run llama3 local server Python client course FAQ'}
iteration #3...
ASSISTANT:
To run Ollama locally:

1. **Install Ollama**
   - **macOS**: download and install the `.pkg` from [ollama.com/download](https://ollama.com/download)
   - **Windows**: download and install the `.msi`
   - **Linux**:
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. **Start a model locally**
   ```bash
   ollama run llama3
   ```
   This will download the model, start it locally, and give you a chat-style prompt.

3. **Check that the local server is running**
   ```bash
   curl http://l

'To run Ollama locally:\n\n1. **Install Ollama**\n   - **macOS**: download and install the `.pkg` from [ollama.com/download](https://ollama.com/download)\n   - **Windows**: download and install the `.msi`\n   - **Linux**:\n     ```bash\n     curl -fsSL https://ollama.com/install.sh | sh\n     ```\n\n2. **Start a model locally**\n   ```bash\n   ollama run llama3\n   ```\n   This will download the model, start it locally, and give you a chat-style prompt.\n\n3. **Check that the local server is running**\n   ```bash\n   curl http://localhost:11434\n   ```\n   If it’s working, you should get a response showing available models.\n\n4. **Use it from Python**\n   ```bash\n   pip install ollama\n   ```\n\n   ```python\n   import ollama\n\n   response = ollama.chat(\n       model=\'llama3\',\n       messages=[{"role": "user", "content": "Hello!"}]\n   )\n\n   print(response[\'message\'][\'content\'])\n   ```\n\nIf you get a connection issue, restarting the server can help:\n```bash\nollama serv